# Description
Run `database_prep.ipynb` before running this script.

# Imports

In [1]:
import pandas as pd
import os
import numpy as np
import statsmodels.api as sm

ModuleNotFoundError: No module named 'statsmodels'

In [35]:
root_dir = "/Users/emmanuelle.coutu-nadeau/Library/Mobile Documents/com~apple~CloudDocs/UdeM/MSc Psycho/LABO NED - Personal Drive/Code/GENiAL/"
og_data = os.path.join(root_dir, 'Data/Final/GENIAL-DB-preprocessed-V2.csv') # original data
og_data = pd.read_csv(og_data)

# Data Manipulations

Make sure EEG features are numerical

In [36]:
# Make sure EEG_ columns are numeric
# Get all EEG_ columns except known non-numeric ones
non_numeric_cols = ['EEG_attempted', 'EEG_site', 'EEG_date', 'EEG_age', 'EEG_Age', 'EEG_Sex']
eeg_cols = [col for col in og_data.columns if col.startswith('EEG_') and col not in non_numeric_cols]

# Convert EEG columns to numeric, coercing errors to NaN
for col in eeg_cols:
    og_data[col] = pd.to_numeric(og_data[col], errors='coerce')


Remove over 80% EEG features issing rows

In [ ]:
# Calculate the percentage of missing values for each row
missing_percentage = og_data[eeg_cols].isnull().mean(axis=1)

# Keep only rows where less than 80% of EEG features are missing
no_missing_data = og_data[missing_percentage < 0.8].reset_index(drop=True)

print(f"Rows remaining after dropping those with >80% missing EEG data: {len(no_missing_data)}")


In [38]:
diagnostic_groups_data = no_missing_data.copy()

# Add diagnostic group column
# 0: Control (diag_control = 1)
# 1: Neurodev only (diag_neurodev = 1 and diag_genetic_carrier = 0)
# 2: Genetic carrier (diag_genetic_carrier = 1)
diagnostic_groups_data['diagnostic_group'] = 0

# Set group 1: Neurodev only
diagnostic_groups_data.loc[(diagnostic_groups_data['diag_neurodev'] == 1) & (diagnostic_groups_data['diag_genetic_carrier'] == 0), 'diagnostic_group'] = 1

# Set group 2: Genetic carrier
diagnostic_groups_data.loc[diagnostic_groups_data['diag_genetic_carrier'] == 1, 'diagnostic_group'] = 2


# Stats

Summarize data

In [ ]:
# Get summary statistics for EEG columns
eeg_summary = diagnostic_groups_data[eeg_cols].agg(['min', 'max', 'mean']).round(2)

# Display summary
print("\nEEG Features Summary:")
print(eeg_summary)


In [ ]:
# Calculate z-scores for each EEG feature within each diagnostic group
z_scores = pd.DataFrame()

for group in diagnostic_groups_data['diagnostic_group'].unique():
    group_data = diagnostic_groups_data[diagnostic_groups_data['diagnostic_group'] == group]
    
    # Calculate z-scores for all EEG columns in this group
    group_z_scores = group_data[eeg_cols].apply(lambda x: (x - x.mean()) / x.std())
    
    # Add group identifier
    group_z_scores['diagnostic_group'] = group
    
    # Append to main z-scores dataframe
    z_scores = pd.concat([z_scores, group_z_scores])

# Reset index of final dataframe
z_scores = z_scores.reset_index(drop=True)

print("\nZ-scores calculated for each diagnostic group")
print(f"Shape of z-scores dataframe: {z_scores.shape}")



In [ ]:
# Check for extreme z-scores (|z| > 3.29)
extreme_mask = (z_scores[eeg_cols].abs() > 3.29)
num_extreme = extreme_mask.sum()

print("\nNumber of extreme z-scores (|z| > 3.29) for each EEG feature:")
print(num_extreme)

# Get columns with extreme scores
columns_with_extremes = [col for col, count in num_extreme.items() if count > 0]
print("\nColumns with extreme scores:")
print(columns_with_extremes)

total_extreme_rows = extreme_mask.any(axis=1).sum()

if total_extreme_rows > 0:
    print(f"\nFound {total_extreme_rows} rows with extreme z-scores")
    print("\nBreakdown by diagnostic group:")
    for group in [0, 1, 2]:
        group_rows = extreme_mask[z_scores['diagnostic_group'] == group].any(axis=1).sum()
        group_name = {
            0: "Control",
            1: "Neurodev only", 
            2: "Genetic carrier"
        }[group]
        print(f"{group_name}: {group_rows} rows with extreme values")
else:
    print("\nNo extreme z-scores found.")


Adjust extreme scores (Z > 3.29)

In [43]:
def adjust_extreme_scores(db, colname):
    # Convert list to array if needed
    if isinstance(db[colname], list):
        db[colname] = np.array(db[colname])
    
    # Convert to numeric
    db[colname] = pd.to_numeric(db[colname])
    
    # Calculate min/max thresholds
    mean = np.nanmean(db[colname])
    std = np.nanstd(db[colname])
    min_val = mean - (3.29 * std)
    max_val = mean + (3.29 * std)
    
    # Replace extreme scores
    db[colname] = np.where(
        pd.isna(db[colname]), 
        np.nan,
        np.where(
            db[colname] < min_val,
            min_val,
            np.where(
                db[colname] > max_val,
                max_val,
                db[colname]
            )
        )
    )
    
    return db

In [44]:
df = diagnostic_groups_data.copy()

# Adjust extreme scores for each column with extreme z-scores
for col in columns_with_extremes:
    df = adjust_extreme_scores(df, col)


Descriptive stats

In [ ]:
# Get numeric columns but exclude binary diagnostic columns
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
numeric_cols = [col for col in numeric_cols if not col.startswith('diag_') and not col.startswith('inheritance_')]
descriptive_stats = df[numeric_cols].describe()

# Add skewness and kurtosis
descriptive_stats.loc['skew'] = df[numeric_cols].skew()
descriptive_stats.loc['kurtosis'] = df[numeric_cols].kurtosis()

# Display the descriptive statistics
print("\nDescriptive Statistics:")
print(descriptive_stats)


In [ ]:
# Prepare data for analysis
# Get all EEG columns
eeg_cols = [col for col in df.columns if col.startswith('EEG_')]

# Define diagnostic groups
diagnostic_groups = [0, 1, 2]  # 0=control, 1=neurodev, 2=genetic carrier
group_labels = ['Control', 'Neurodevelopmental', 'Genetic Carrier']

# Convert EEG columns to numeric type and handle any string values
for col in eeg_cols:
    # Replace any non-numeric values with NaN
    df[col] = pd.to_numeric(df[col].replace(['', 'NA', 'nan'], np.nan), errors='coerce')
    
    # Print number of valid values to verify data
    valid_count = df[col].notna().sum()
    print(f"{col}: {valid_count} valid numeric values")

# Perform one-way ANOVA for each EEG feature
from scipy import stats
anova_results = {}

print("\nOne-way ANOVA Results:")
print("-" * 50)

for eeg_feature in eeg_cols:
    # Create lists to hold the data for each diagnostic group
    groups_data = [df[df['diagnostic_group'] == group][eeg_feature].dropna().values for group in diagnostic_groups]
    
    # Print group sizes to debug
    print(f"\nAnalyzing {eeg_feature}")
    for group, label, data in zip(diagnostic_groups, group_labels, groups_data):
        print(f"{label} group size: {len(data)}")

    # Skip if any group has no data
    if any(len(group) == 0 for group in groups_data):
        print(f"Skipping {eeg_feature} - insufficient data in one or more groups")
        continue
        
    try:
        # Perform one-way ANOVA
        f_stat, p_val = stats.f_oneway(*groups_data)
        
        # Store results
        anova_results[eeg_feature] = {
            'F-statistic': f_stat,
            'p-value': p_val
        }
        
        # Print results
        print(f"\nFeature: {eeg_feature}")
        print(f"F-statistic: {f_stat:.4f}")
        print(f"p-value: {p_val:.4f}")
        
        # If significant, perform post-hoc t-tests
        if p_val < 0.05:
            print("\nPost-hoc t-tests:")
            for i, (group1, label1) in enumerate(zip(diagnostic_groups, group_labels)):
                for group2, label2 in zip(diagnostic_groups[i+1:], group_labels[i+1:]):
                    group1_data = df[df['diagnostic_group'] == group1][eeg_feature].dropna()
                    group2_data = df[df['diagnostic_group'] == group2][eeg_feature].dropna()
                    
                    if len(group1_data) > 0 and len(group2_data) > 0:
                        t_stat, t_pval = stats.ttest_ind(group1_data, group2_data)
                        print(f"{label1} vs {label2}:")
                        print(f"t-statistic: {t_stat:.4f}")
                        print(f"p-value: {t_pval:.4f}")
                    else:
                        print(f"Skipping {label1} vs {label2} - insufficient data")
                        
    except Exception as e:
        print(f"\nError analyzing {eeg_feature}: {str(e)}")

# Multiple regression for each EEG feature
import statsmodels.api as sm

print("\nMultiple Regression Results:")
print("-" * 50)

# Create dummy variables for diagnostic groups
diagnostic_dummies = pd.get_dummies(df['diagnostic_group'], prefix='group')

for eeg_feature in eeg_cols:
    try:
        # Prepare X (predictors) and y (dependent variable)
        X = diagnostic_dummies
        y = pd.to_numeric(df[eeg_feature], errors='coerce')
        
        # Remove rows with NaN values
        mask = ~y.isna()
        X = X[mask]
        y = y[mask]
        
        if len(y) == 0:
            print(f"\nSkipping regression for {eeg_feature} - no valid data")
            continue
            
        # Add constant
        X = sm.add_constant(X)
        
        # Fit model
        model = sm.OLS(y, X).fit()
        
        # Print results
        print(f"\nDependent Variable: {eeg_feature}")
        print(model.summary().tables[1])
        
    except Exception as e:
        print(f"\nError in regression for {eeg_feature}: {str(e)}")
